In [ ]:
from pypdf import PdfReader
import os, re, shutil

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio",
          "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
MES_PATTERN = "|".join(MESES)

def extract_meeting_date(pdf_path):
    reader = PdfReader(pdf_path)
    text = reader.pages[0].extract_text().lower()

    # Primary: the title itself, "comunicado rpm {mes} {año}"
    m = re.search(rf"comunicado\s+rpm\s+({MES_PATTERN})\s+(20\d\d)", text)
    if m:
        return m.group(2), m.group(1)

    # Fallback: the dateline, "{día} de {mes} de {año}"
    m = re.search(rf"\d{{1,2}}\s+de\s+({MES_PATTERN})\s+de\s+(20\d\d)", text)
    if m:
        return m.group(2), m.group(1)

    return None, None


def organize_by_content(source_dir, output_dir="pdfs", min_year=2016):
    os.makedirs(output_dir, exist_ok=True)
    renamed, unmatched = [], []

    for filename in os.listdir(source_dir):
        if not filename.lower().endswith(".pdf"):
            continue
        src_path = os.path.join(source_dir, filename)
        try:
            year, mes = extract_meeting_date(src_path)
        except Exception as e:
            unmatched.append((filename, f"extraction error: {e}"))
            continue

        if not (year and mes):
            unmatched.append((filename, "no title/dateline match"))
            continue

        if int(year) < min_year:
            unmatched.append((filename, f"pre-2020 ({year}) — out of scope, skipped"))
            continue

        new_name = f"{year}_{mes}.pdf"
        dst_path = os.path.join(output_dir, new_name)
        if os.path.exists(dst_path):
            unmatched.append((filename, f"collision with {new_name} — check both"))
            continue

        shutil.copy2(src_path, dst_path)
        renamed.append((filename, new_name))

    print(f"{len(renamed)} renamed, {len(unmatched)} unmatched\n")
    for old, new in renamed:
        print(f"  {old} -> {new}")
    if unmatched:
        print(f"\nunmatched:")
        for f, reason in unmatched:
            print(f"  {f}: {reason}")

    return renamed, unmatched


renamed, unmatched = organize_by_content(source_dir=r"C:\DS_ML\machine-learning-engineer\monetary-policy-rag\data",output_dir=r"C:\DS_ML\machine-learning-engineer\monetary-policy-rag\data\chile_banco_central_2")

Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 57 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 62 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 65 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)


90 renamed, 3 unmatched

  07122020-RPM.pdf -> 2020_diciembre.pdf
  13052021_comunicaco_rpm.pdf -> 2021_mayo.pdf
  13octubre2021_Comunicado_RPM.pdf -> 2021_octubre.pdf
  27012021_comunicado_RPM.pdf -> 2021_enero.pdf
  29-03-2022RPM_Comunicado.pdf -> 2022_marzo.pdf
  30032021_rpm.pdf -> 2021_marzo.pdf
  bcch_comunicado_104040_es.pdf -> 2016_enero.pdf
  bcch_comunicado_104433_es.pdf -> 2016_febrero.pdf
  bcch_comunicado_160884_es.pdf -> 2016_marzo.pdf
  bcch_comunicado_161120_es.pdf -> 2016_abril.pdf
  bcch_comunicado_161389_es.pdf -> 2016_mayo.pdf
  bcch_comunicado_164760_es.pdf -> 2016_junio.pdf
  bcch_comunicado_164967_es.pdf -> 2016_julio.pdf
  bcch_comunicado_165150_es.pdf -> 2016_agosto.pdf
  bcch_comunicado_168273_es.pdf -> 2016_septiembre.pdf
  bcch_comunicado_169989_es.pdf -> 2016_noviembre.pdf
  bcch_comunicado_170380_es.pdf -> 2016_diciembre.pdf
  bcch_comunicado_170923_es.pdf -> 2017_enero.pdf
  bcch_comunicado_171258_es.pdf -> 2017_febrero.pdf
  bcch_comunicado_171486_es.pdf

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import os

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio",
         "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

AÑOS = range(2016, 2027)  # 2020–2026 inclusive

def scrape_comunicados(output_dir="pdfs"):
    os.makedirs(output_dir, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0 (research project; contact: your_email@example.com)"}

    hits, misses = [], []

    for año in AÑOS:
        for mes in MESES:
            url = f"https://www.bcentral.cl/web/banco-central/contenido/-/detalle/prensa/comunicados-rpm/comunicado-rpm-{mes}-{año}"
            try:
                resp = requests.get(url, headers=headers, timeout=15)
                if resp.status_code != 200:
                    misses.append((mes, año, resp.status_code))
                    time.sleep(1.0)
                    continue

                soup = BeautifulSoup(resp.text, "html.parser")
                print(soup)
                pdf_link = None
                for a in soup.find_all("a", href=True):
                    print("found ", a)
                    if ".pdf" in a["href"].lower():
                        pdf_link = a["href"]
                        break

                if not pdf_link:
                    misses.append((mes, año, "no_pdf_found"))
                    time.sleep(1.0)
                    continue

                pdf_resp = requests.get(pdf_link, headers=headers, timeout=15)
                filename = f"{output_dir}/comunicado_{año}_{mes}.pdf"
                with open(filename, "wb") as f:
                    f.write(pdf_resp.content)
                hits.append((mes, año))
                print(f"OK: {mes} {año}")

            except Exception as e:
                misses.append((mes, año, str(e)))

            time.sleep(1.5)

    print(f"\n{len(hits)} downloaded, {len(misses)} misses")
    return hits, misses

hits, misses = scrape_comunicados(output_dir=r"C:\DS_ML\machine-learning-engineer\monetary-policy-rag\data")

<html style="height:100%"><head><meta content="NOINDEX, NOFOLLOW" name="ROBOTS"/><meta content="telephone=no" name="format-detection"/><meta content="initial-scale=1.0" name="viewport"/><meta content="IE=edge,chrome=1" http-equiv="X-UA-Compatible"/></head><body style="margin:0px;height:100%"><iframe frameborder="0" height="100%" id="main-iframe" marginheight="0px" marginwidth="0px" src="/_Incapsula_Resource?SWUDNSAI=31&amp;xinfo=45-864517-0%200NNN%20RT%281787173165182%2022%29%20q%280%20-1%20-1%20-1%29%20r%280%20-1%29%20B12%284%2c315%2c0%29%20U12&amp;incident_id=529000060036871668-4330417990730157&amp;edet=12&amp;cinfo=04000000&amp;rpinfo=0&amp;cts=VYUnSqqMqoLIp%2btAu1ZDKEp6nAGBYk61NA8mfnVXGsxISM9pxX6oV1fXdDGIiPmW&amp;cip=200.9.3.65&amp;mth=GET" width="100%">Request unsuccessful. Incapsula incident ID: 529000060036871668-4330417990730157</iframe></body></html>
<html style="height:100%"><head><meta content="NOINDEX, NOFOLLOW" name="ROBOTS"/><meta content="telephone=no" name="format-detec

KeyboardInterrupt: 